<a id='1'></a>
## 1 - Loading the libraries

In [61]:
import json
# import cohere
from google import genai
import os
from dotenv import load_dotenv
import time
from classifier import language_inference
load_dotenv()

True

<a id='2'></a>
## 2 - Setting Configurations

In [62]:
BASE_RESPONSES = {
    "greeting": "Hello 👋 How can I help you today?",
    "goodbye": "Take care. Have a great day.",
    "gratitude": "You're welcome. I'm glad I could help.",
    "mental_health": "I'm here to listen. Would you like to talk more about what you're feeling?",
    "out_of_scope": "I'm designed mainly for mental health support, greetings, and basic conversational interactions."
}

In [63]:
# CONSTANTS
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
MODEL_NAME = "aya-expanse-32b"
INTENTS = [
    "greeting",
    "goodbye",
    "gratitude",
    "asking_mental_health_question",
    "out_of_scope"
]

CONFIDENCE_THRESHOLD = 0.65

predictor = language_inference.LanguagePredictor()

d:\ITI Materials\Projects\Health-Rag-System\.venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\ITI Materials\Projects\Health-Rag-System\.venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\ITI Materials\Projects\Health-Rag-System\.venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LogisticR

In [64]:
predictor.predict("hola amigo")

np.str_('sw')

In [65]:
# Client
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

In [66]:
# SYSTEM PROMPT
def build_intent_classifier_prompt(user_message: str) -> str:

    return f"""
You are an intent classification engine.

Your task is to classify the user's message into EXACTLY ONE intent.

Possible intents:

1. greeting
Definition:
The user is greeting, saying hello, starting a conversation,
or making a casual salutation.

2. goodbye
Definition:
The user is ending the conversation or saying farewell.

3. gratitude
Definition:
The user is thanking, appreciating,
or expressing gratitude.

4. asking_mental_health_question
Definition:
The user is discussing emotions, sadness, anxiety,
depression, stress, loneliness, emotional struggles,
mental health concerns, or asking psychological support questions.

Includes:
- depression
- anxiety
- panic
- loneliness
- stress
- emotional pain
- hopelessness
- emotional support seeking

5. out_of_scope
Definition:
The message does not belong to any previous category.

Rules:
- Always return ONLY valid JSON.
- Never explain.
- Never output markdown.
- Choose exactly one intent.
- Detect the intent regardless of language.
- Handle mixed-language text.
- Consider semantic meaning, not keywords only.

Allowed intents:
{INTENTS}

Examples:

User: "hello"
Output:
{{"intent":"greeting","confidence":0.99}}

User: "مرحبا كيف حالك"
Output:
{{"intent":"greeting","confidence":0.98}}

User: "hola amigo"
Output:
{{"intent":"greeting","confidence":0.98}}

User: "goodbye my friend"
Output:
{{"intent":"goodbye","confidence":0.99}}

User: "مع السلامة"
Output:
{{"intent":"goodbye","confidence":0.98}}

User: "thanks for helping me"
Output:
{{"intent":"gratitude","confidence":0.99}}

User: "شكرا جدا"
Output:
{{"intent":"gratitude","confidence":0.98}}

User: "I feel depressed and lonely"
Output:
{{"intent":"asking_mental_health_question","confidence":0.97}}

User: "انا حاسس باكتئاب"
Output:
{{"intent":"asking_mental_health_question","confidence":0.98}}

User: "انا تعبان نفسيا today"
Output:
{{"intent":"asking_mental_health_question","confidence":0.97}}

User: "what is the weather today"
Output:
{{"intent":"out_of_scope","confidence":0.99}}

Now classify this message:

User: "{user_message}"
"""

In [67]:
# LLM Call
def call_llm(prompt: str) -> dict:

    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config={
            "temperature": 0,
            "response_mime_type": "application/json"
        }
    )

    return json.loads(response.text)

In [68]:
def validate_intent_response(response: dict) -> dict:

    if "intent" not in response:
        return {
            "intent": "out_of_scope",
            "confidence": 0.0
        }

    if "confidence" not in response:
        response["confidence"] = 0.0

    if response["intent"] not in INTENTS:
        return {
            "intent": "out_of_scope",
            "confidence": 0.0
        }

    return response

In [69]:
# INTENT CLASSIFICATION
def classify_intent(user_message: str) -> dict:

    prompt = build_intent_classifier_prompt(user_message)

    llm_response = call_llm(prompt)

    validated_response = validate_intent_response(llm_response)

    language = predictor.predict(user_message)

    validated_response["language"] = language

    return validated_response

In [70]:
def apply_confidence_threshold(intent_data: dict) -> dict:

    confidence = intent_data["confidence"]

    if confidence < CONFIDENCE_THRESHOLD:
        intent_data["intent"] = "out_of_scope"
        intent_data["confidence"] = confidence

    return intent_data

In [71]:
# Translate
def translate(text: str, target_language: str):

    prompt = f"""
Translate the following text into {target_language}.

Rules:
- Keep meaning exactly the same
- Keep tone natural and empathetic
- Do NOT add extra information
- Return only the translated text

Text:
{text}
"""

    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config={"temperature": 0}
    )

    return response.text.strip()

# Static Response
def build_response(intent: str, language: str):

    base_text = BASE_RESPONSES[intent]

    # If English → no translation needed
    if language == "en":
        return base_text

    return translate(base_text, language)

In [72]:
# Hanlder
def greeting_handler(user_message: str, language: str):
    return build_response("greeting", language)


def goodbye_handler(user_message: str, language: str):
    return build_response("goodbye", language)


def gratitude_handler(user_message: str, language: str):
    return build_response("gratitude", language)


def mental_health_handler(user_message: str, language: str):
    return build_response("mental_health", language)


def out_of_scope_handler(user_message: str, language: str):
    return build_response("out_of_scope", language)

In [73]:
# =========================================================
# MAIN ROUTER
# =========================================================

def route_intent(user_message: str):

    # Step 1
    intent_data = classify_intent(user_message)

    # Step 2
    intent_data = apply_confidence_threshold(intent_data)

    intent = intent_data["intent"]
    language = intent_data["language"]

    # Step 3
    if intent == "greeting":
        return greeting_handler(user_message, language)

    elif intent == "goodbye":
        return goodbye_handler(user_message, language)

    elif intent == "gratitude":
        return gratitude_handler(user_message, language)

    elif intent == "mental_health":
        return mental_health_handler(user_message, language)

    else:
        return out_of_scope_handler(user_message, language)

In [74]:
# FULL PIPELINE
# =========================================================
def chatbot(user_message: str):

    response = route_intent(user_message)

    return response

In [75]:
examples = [
    "hello",
    "مرحبا",
    "hola amigo",
    "شكرا جدا",
    "I feel anxious and lonely",
    "انا تعبان نفسيا",
    "what is the weather today",
    "bye"
]

for msg in examples:

    print("=" * 50)

    print("USER:", msg) 

    result = classify_intent(msg)

    print("INTENT:", result)

    final_response = chatbot(msg)

    print("BOT:", final_response)

    time.sleep(3)

USER: hello
INTENT: {'intent': 'greeting', 'confidence': 0.99, 'language': np.str_('it')}
BOT: Ciao 👋 Come posso aiutarti oggi?
USER: مرحبا
INTENT: {'intent': 'greeting', 'confidence': 0.99, 'language': np.str_('ar')}
BOT: مرحباً، كيف يمكنني مساعدتك اليوم؟
USER: hola amigo


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 24.374640014s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'